In [1]:
from src.config import BENCHMARK_PATH  # Dataset

from src.vector_store import load_vector_store  # RAG Index
from src.rag import create_audit_client, run_audits  # LLM Audit
from src.evaluation import load_benchmark, evaluate  # Benchmark

from src.ollama_server import setup_ollama  # LLM Server

## Load Ollama and Vector Store

In [2]:
setup_ollama()

Ollama server is already running.
nomic-embed-text is already installed.
qwen3:4b is already installed.
Required Ollama models are ready.


In [3]:
vector_store = load_vector_store()
client = create_audit_client()

## Load Benchmark

In [4]:
benchmark, requirements = load_benchmark(BENCHMARK_PATH)
print(f"Loaded {len(requirements)} requirements")
display(benchmark.head())
display(benchmark["label"].value_counts())

Loaded 50 requirements


,requirement_id,swe_id,requirement_text,clause_id,label,rationale
0,1,SWE-050,The system shall document all functional and C...,4.1.2,Meets,"Requirements, including those for COTS compone..."
1,2,SWE-051,The software requirements shall be derived fro...,4.1.3,Partial,Requirements analysis traces to systems engine...
2,3,SWE-053,Requirement changes shall be recorded in a spr...,4.1.5,Partial,"Changes are logged, satisfying basic tracking,..."
3,4,SWE-055,The system shall undergo requirements validati...,4.1.7,Meets,Validation activity confirms the software will...
4,5,SWE-057,The software architecture document shall be pr...,4.2.3,Partial,"An architecture artifact exists, but the requi..."


label
Meets      22
Partial    14
Gap        14
Name: count, dtype: int64

In [5]:
limit = None  # Use None for the full benchmark.
retrievals = 5

evaluation_set = benchmark.iloc[:limit].copy()
evaluation_requirements = requirements[:limit]

## Audit and Evaluate Benchmark

In [6]:
records = run_audits(evaluation_requirements, vector_store, client, k=retrievals)

[1/50] 1: Meets
[2/50] 2: Partial
[3/50] 3: Gap
[4/50] 4: Meets
[5/50] 5: Meets
[6/50] 6: Meets
[7/50] 7: Meets
[8/50] 8: Gap
[9/50] 9: Meets
[10/50] 10: Meets
[11/50] 11: Meets
[12/50] 12: Meets
[13/50] 13: Meets
[14/50] 14: Meets
[15/50] 15: Meets
[16/50] 16: Meets
[17/50] 17: FAILED
[18/50] 18: Partial
[19/50] 19: Meets
[20/50] 20: Gap
[21/50] 21: Gap
[22/50] 22: Meets
[23/50] 23: Meets
[24/50] 24: Meets
[25/50] 25: Gap
[26/50] 26: Partial
[27/50] 27: Gap
[28/50] 28: Gap
[29/50] 29: Partial
[30/50] 30: Gap
[31/50] 31: Meets
[32/50] 32: Gap
[33/50] 33: Meets
[34/50] 34: Partial
[35/50] 35: Meets
[36/50] 36: Meets
[37/50] 37: Gap
[38/50] 38: Gap
[39/50] 39: Meets
[40/50] 40: Partial
[41/50] 41: Meets
[42/50] 42: Gap
[43/50] 43: Gap
[44/50] 44: Gap
[45/50] 45: Meets
[46/50] 46: Partial
[47/50] 47: Gap
[48/50] 48: Meets
[49/50] 49: Gap
[50/50] 50: Gap


In [7]:
rows, metrics, matrix = evaluate(
    evaluation_set,
    records,
    k=retrievals,
)

print(f"Requirements evaluated: {metrics['n']}")
print(f"Verdict accuracy: {metrics['verdict_accuracy']:.1%}")
print(f"Macro F1: {metrics['macro_f1']:.4f}")
print(
    f"Retrieval Hit@{retrievals}: "
    f"{metrics['retrieval'][f'hit_at_{retrievals}']:.1%}"
)

display(matrix)
display(rows)

Requirements evaluated: 50
Verdict accuracy: 74.0%
Macro F1: 0.7005
Retrieval Hit@5: 98.0%


model_verdict,Meets,Partial,Gap,INVALID
human_verdict,,,,
Meets,20,2,0,0
Partial,4,5,5,0
Gap,1,0,12,1


,requirement_id,swe_id,requirement_text,clause_id,label,rationale,pred_verdict,pred_clause,retrieved,hit_at_1,hit_at_k,citation_member,correct
0,1,SWE-050,The system shall document all functional and C...,4.1.2,Meets,"Requirements, including those for COTS compone...",Meets,4.1.2,"4.1.2,4.1.3,2.2.8,4.1.4,2.1.5.12",True,True,True,True
1,2,SWE-051,The software requirements shall be derived fro...,4.1.3,Partial,Requirements analysis traces to systems engine...,Partial,4.1.3,"4.1.2,4.1.3,2.1.8.2,3.1.13,4.1.4",False,True,True,True
2,3,SWE-053,Requirement changes shall be recorded in a spr...,4.1.5,Partial,"Changes are logged, satisfying basic tracking,...",Gap,4.1.5,"4.1.5,2.2.7,4.1.2,3.5.2,2.1.8.2",True,True,True,False
3,4,SWE-055,The system shall undergo requirements validati...,4.1.7,Meets,Validation activity confirms the software will...,Meets,4.1.7,"4.1.7,4.1.2,4.5.12,4.1.3,4.5.8",True,True,True,True
4,5,SWE-057,The software architecture document shall be pr...,4.2.3,Partial,"An architecture artifact exists, but the requi...",Meets,4.2.3,"4.2.3,4.3.2,3.1.6,4.1.2,4.2.4",True,True,True,False
5,6,SWE-058,The detailed design document shall describe ea...,4.3.2,Meets,Design is recorded at a level of detail enabli...,Meets,4.3.2,"4.3.2,4.4.5,4.5.13,4.2.3,4.5.8",True,True,True,True
6,7,SWE-060,Developers shall write source code implementin...,4.4.2,Meets,Code implementation is explicitly tied to the ...,Meets,4.4.2,"4.3.2,4.4.2,3.1.10,4.4.8,3.1.8",False,True,True,True
7,8,SWE-061,All source code for the project shall be writt...,4.4.3,Gap,"Specifying a language does not select, define,...",Gap,3.8.1,"3.8.1,3.8.2,4.4.2,3.1.10,4.4.3",False,True,True,True
8,9,SWE-135,Static analysis shall be performed on all sour...,4.4.4,Partial,"Defect detection is covered, but security, cod...",Meets,3.11.7,"3.11.7,4.4.4,3.11.5,5.5.3,4.5.10",False,True,True,False
9,10,SWE-062,Each software unit shall be unit tested prior ...,4.4.5,Meets,Unit-level testing is explicitly required befo...,Meets,4.4.5,"4.4.5,4.5.4,4.5.13,4.5.3,4.5.14",True,True,True,True


In [8]:
display(rows.loc[~rows["correct"]])

for record in records:
    if record.error:
        print(f"{record.requirement_id}: {record.error}")

,requirement_id,swe_id,requirement_text,clause_id,label,rationale,pred_verdict,pred_clause,retrieved,hit_at_1,hit_at_k,citation_member,correct
2,3,SWE-053,Requirement changes shall be recorded in a spr...,4.1.5,Partial,"Changes are logged, satisfying basic tracking,...",Gap,4.1.5,"4.1.5,2.2.7,4.1.2,3.5.2,2.1.8.2",True,True,True,False
4,5,SWE-057,The software architecture document shall be pr...,4.2.3,Partial,"An architecture artifact exists, but the requi...",Meets,4.2.3,"4.2.3,4.3.2,3.1.6,4.1.2,4.2.4",True,True,True,False
8,9,SWE-135,Static analysis shall be performed on all sour...,4.4.4,Partial,"Defect detection is covered, but security, cod...",Meets,3.11.7,"3.11.7,4.4.4,3.11.5,5.5.3,4.5.10",False,True,True,False
12,13,SWE-066,The software shall pass system-level tests bef...,4.5.3,Gap,The requirement does not tie testing to the so...,Meets,4.5.8,"4.5.8,4.5.3,4.4.5,4.5.13,4.5.12",False,True,True,False
15,16,SWE-189,Code coverage shall be measured using a covera...,4.5.9,Partial,"Measurement occurs, but tracking, recording, a...",Meets,4.5.10,"4.5.10,4.5.9,3.7.4,4.4.5,5.4.5",False,True,True,False
16,17,SWE-190,Code coverage percentage shall be calculated b...,4.5.10,Gap,The clause requires coverage measured through ...,INVALID,,"4.5.10,4.5.9,3.7.4,5.4.5,4.4.4",True,True,False,False
19,20,SWE-193,Uplinked command tables shall be validated aga...,4.5.13,Partial,Checksum validation addresses data integrity b...,Gap,5.5.3,"5.5.3,3.1.13,3.6.1,2.1.8.2,2.1.5.4",False,False,True,False
22,23,SWE-154,Cybersecurity risks for the ground software sy...,3.11.3,Partial,Risk identification and mitigation are documen...,Meets,3.11.3,"3.11.3,5.2,3.11.2,3.11.5,3.11.4",True,True,True,False
25,26,SWE-207,Developers shall follow secure coding guidelin...,3.11.6,Meets,"Secure coding practices are identified, record...",Partial,3.11.7,"3.11.7,3.11.6,4.4.3,3.7.4,4.4.4",False,True,True,False
27,28,SWE-210,The system shall log all failed login attempts...,3.11.8,Partial,Some adversarial-action data is collected and ...,Gap,3.11.8,"5.5.3,5.5.2,3.11.8,5.5.1,4.5.11",False,True,True,False


17: Citation validation failed: Evidence must be an exact excerpt of the selected clause's requirement
